# Market Data Wrangling and Visualization (Identifying whales, $\Delta$ price, etc)

## Set Up

In [1]:
import time
import requests
import pandas as pd
import numpy as np
import altair as alt

In [2]:
market_test = pd.read_csv("market_csvs/trumps-31_trades.csv")
market_test = market_test.drop(columns = ["asset","conditionId","icon","eventSlug","name", "pseudonym", "bio", "profileImage", "profileImageOptimized", "transactionHash"])
market_test.head()

,proxyWallet,side,size,price,timestamp,title,slug,outcome,outcomeIndex
0,0xec4e8e525969c4b9072eed0193e587b73876d9db,SELL,360.400,0.999000,1769930976,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,No,1
1,0x5f94ea09f1fe4b506a785ac4526e27a9391f1454,SELL,2.000,0.999000,1769929438,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,No,1
2,0x9380db17dd5ab2ce55301effa758fd71a6e501ed,BUY,174.000,0.996731,1769890322,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,No,1
3,0x89f0a333cda79fccc920dc0f003823e9bece1e62,BUY,3.006,0.998000,1769858808,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,No,1
4,0x272df07434614e735c7896b0eb9d3d26c78d9e36,BUY,3.006,0.998000,1769858798,Trump's travel ban removed by January 31?,trumps-travel-ban-removed-by-january-31,No,1


## Identifying Whales

In [3]:
# wallet statistic summary
wallet_stats = market_test.groupby("proxyWallet").agg(
    total_volume=("size", "sum"),
    n_trades=("size", "count"),
    avg_price=("price", "mean"),
    largest_trade=("size", "max"),
    first_trade=("timestamp", "min"),
    last_trade=("timestamp", "max")
)

# sort by size
wallet_stats = wallet_stats.sort_values("total_volume", ascending=False)
wallet_stats = wallet_stats.reset_index()
wallet_stats

# define thresholds for whales
market_volume = market_test["size"].sum()
volume_threshold = wallet_stats["total_volume"].quantile(0.95)
trade_threshold = wallet_stats["largest_trade"].quantile(0.95)


# whales can take two types: single traes and multiple trades, for now sort both
whales_df = wallet_stats[
    (wallet_stats["total_volume"] > volume_threshold) |
    (wallet_stats["largest_trade"] > trade_threshold)
]

print(volume_threshold, trade_threshold)
whales_df

434.0 326.0


,proxyWallet,total_volume,n_trades,avg_price,largest_trade,first_trade,last_trade
0,0x977449de8b5be74e089f3b28f035ebf3e5f46d3c,1510.000000,3,0.870000,755.0000,1766007929,1766395265
1,0xec4e8e525969c4b9072eed0193e587b73876d9db,1040.804887,8,0.941401,360.4000,1766264775,1769930976
2,0xd14aa86fa6cac2e09edd9944f5c0c0d48634a8ad,927.877100,1,0.969956,927.8771,1769232200,1769232200
3,0x60ccc2ea4a54b191b6114fa769c03359bb1af23f,669.000000,1,0.150000,669.0000,1766058414,1766058414
4,0xeaf1f1805537078c833c2acba571b1dd765668ca,635.000000,3,0.870000,326.0000,1766006877,1766395441
9,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,387.400000,2,0.043210,386.3100,1767440993,1769248852


## Plotting Price / Time

In [4]:
df = market_test.copy()

# convert price to be measured only for winning outcome (index = 0)
df["price_outcome"] = df["price"]
df.loc[df["outcomeIndex"] == 0, "price_outcome"] = (1 - df.loc[df["outcomeIndex"] == 0, "price"])

# parse time + clean
df = df.sort_values("timestamp")
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
df = df.dropna(subset=["timestamp", "price_outcome"]).sort_values("timestamp")

# plot graph
market_graph = alt.Chart(df).mark_line().encode(
    alt.X("timestamp:T").title("Time"),
    alt.Y("price_outcome:Q").title("Price (USD)")
).properties(width = 600, height = 300, title="$ over time of Resolution Outcome")

market_graph

alt.Chart(...)

## Watching Whales

In [5]:
df["is_whale"] = df["proxyWallet"].isin(whales_df["proxyWallet"])

whale_points = alt.Chart(df[df["is_whale"]]).mark_point(filled=True, size=60, color="Red").encode(
    alt.X("timestamp:T").title("Time"),
    alt.Y("price_outcome:Q").title("Price (USD)"),
    tooltip=[alt.Tooltip("proxyWallet:N", title="Wallet ID:"),
             alt.Tooltip("outcomeIndex:Q", title="Outcome:"),
             alt.Tooltip("side:N", title="Action:")]
)

chart = (market_graph + whale_points).properties(width = 600, height = 300, title="$ over time of Resolution Outcome with Whale Transactions")


chart

alt.LayerChart(...)

# Watching the Money

In [6]:
# Now want to try to calculate changes in price for each of these flagged points

# Clean up trades for only whales
whale_trades = df[df["is_whale"]]
whale_trades = whale_trades.sort_values("proxyWallet")
whale_trades

# Claude
base = df[["timestamp", "price_outcome"]].sort_values("timestamp")
def price_after(horizon: pd.Timedelta) -> pd.Series:
    t = whale_trades[["timestamp"]].copy()
    t["target_time"] = t["timestamp"] + horizon
    t["orig_idx"] = t.index  # ← preserve original whale_trades index as-is

    merged = pd.merge_asof(
        t.sort_values("target_time"),
        base.sort_values("timestamp"),
        left_on="target_time",
        right_on="timestamp",
        direction="nearest",
        allow_exact_matches=True,
    )

    merged = merged.sort_values("orig_idx").set_index("orig_idx")
    return merged["price_outcome"]  # ← now indexed same as whale_trades

# Columns to DF
whale_trades["p_t"] = whale_trades["price_outcome"]

# prices after fixed horizons
whale_trades["p_1h"]  = price_after(pd.Timedelta(hours=1))
whale_trades["p_6h"]  = price_after(pd.Timedelta(hours=6))
whale_trades["p_1d"]  = price_after(pd.Timedelta(days=1))
whale_trades["p_3d"]  = price_after(pd.Timedelta(days=3))

# deltas vs trade-time price
whale_trades["Change after 1h"] = whale_trades["p_1h"] - whale_trades["p_t"]
whale_trades["Change after 6h"] = whale_trades["p_6h"] - whale_trades["p_t"]
whale_trades["Change after 1d"] = whale_trades["p_1d"] - whale_trades["p_t"]
whale_trades["Change after 3d"] = whale_trades["p_3d"] - whale_trades["p_t"]

# Next Whale
wt = whale_trades.sort_values("timestamp")
wt["p_next_whale"] = wt["price_outcome"].shift(-1)
wt["Change until next Whale"] = wt["p_next_whale"] - wt["price_outcome"]
wt["hours_until_next_whale"] = (wt["timestamp"].shift(-1) - wt["timestamp"]).dt.total_seconds() / 3600
whale_trades = wt

whale_trades


# Create DF
whale_trade_table = whale_trades[[
    "timestamp",
    "proxyWallet",
    "side",
    "size",
    "price",     
    "outcome",
    "outcomeIndex",
    "price_outcome",
    "Change after 1h",
    "Change after 6h",
    "Change after 1d",
    "Change after 3d",
    "Change until next Whale",
]].reset_index(drop=True)

whale_trade_table = whale_trade_table.round(4)

whale_trade_table


,timestamp,proxyWallet,side,size,price,outcome,outcomeIndex,price_outcome,Change after 1h,Change after 6h,Change after 1d,Change after 3d,Change until next Whale
0,2025-12-17 21:27:57,0xeaf1f1805537078c833c2acba571b1dd765668ca,BUY,326.0000,0.8300,No,1,0.8300,0.0078,0.0300,0.0300,0.0700,0.0000
1,2025-12-17 21:45:29,0x977449de8b5be74e089f3b28f035ebf3e5f46d3c,BUY,755.0000,0.8300,No,1,0.8300,0.0078,0.0300,0.0300,0.0700,0.0200
2,2025-12-18 11:46:54,0x60ccc2ea4a54b191b6114fa769c03359bb1af23f,BUY,669.0000,0.1500,Yes,0,0.8500,0.0000,0.0100,0.0309,0.0400,0.0500
3,2025-12-20 21:06:15,0xec4e8e525969c4b9072eed0193e587b73876d9db,BUY,160.0000,0.9000,No,1,0.9000,0.0000,0.0000,0.0000,-0.0000,-0.0100
4,2025-12-22 09:19:37,0x977449de8b5be74e089f3b28f035ebf3e5f46d3c,SELL,542.0000,0.8900,No,1,0.8900,0.0000,0.0100,0.0100,0.0100,0.0000
5,2025-12-22 09:21:05,0x977449de8b5be74e089f3b28f035ebf3e5f46d3c,SELL,213.0000,0.8900,No,1,0.8900,0.0000,0.0100,0.0100,0.0100,0.0000
6,2025-12-22 09:22:51,0xeaf1f1805537078c833c2acba571b1dd765668ca,SELL,113.0000,0.8900,No,1,0.8900,0.0000,0.0100,0.0100,0.0100,0.0000
7,2025-12-22 09:24:01,0xeaf1f1805537078c833c2acba571b1dd765668ca,SELL,196.0000,0.8900,No,1,0.8900,0.0000,0.0100,0.0100,0.0100,0.0100
8,2025-12-22 11:56:05,0xec4e8e525969c4b9072eed0193e587b73876d9db,BUY,48.8889,0.9000,No,1,0.9000,0.0000,0.0000,0.0000,0.0000,0.0000
9,2025-12-31 12:00:11,0xec4e8e525969c4b9072eed0193e587b73876d9db,BUY,222.2222,0.9000,No,1,0.9000,0.0000,0.0000,0.0010,0.0200,0.0010


In [7]:
summary_stats = whale_trade_table.describe().round(4)

summary_stats

,timestamp,size,price,outcomeIndex,price_outcome,Change after 1h,Change after 6h,Change after 1d,Change after 3d,Change until next Whale
count,18,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,17.0000
mean,2026-01-03 19:33:19.444444416,287.2268,0.7754,0.8333,0.9158,0.0009,0.0069,0.0082,0.0188,0.0099
min,2025-12-17 21:27:57,1.0900,0.0064,0.0000,0.8300,0.0000,-0.0100,-0.0225,-0.0146,-0.0100
25%,2025-12-22 09:19:59,70.0000,0.8450,1.0000,0.8900,0.0000,0.0000,0.0000,0.0000,0.0000
50%,2025-12-26 23:58:08,204.5000,0.8950,1.0000,0.9000,0.0000,0.0002,0.0095,0.0100,0.0010
75%,2026-01-23 11:57:59.500000,379.8325,0.9378,1.0000,0.9650,0.0000,0.0100,0.0175,0.0298,0.0200
max,2026-02-01 07:29:36,927.8771,0.9990,1.0000,0.9990,0.0078,0.0300,0.0309,0.0700,0.0500
std,NaN,272.7395,0.3252,0.3835,0.0548,0.0025,0.0112,0.0145,0.0239,0.0153


![Whale Watching](other/whalewatching.jpg)

This also works! some parts are a little funky and will have to clean up the visualization, time to move on to making all of this work coherently and do this for every market!